In [1]:
"""Main file for training DiffCSP."""

import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, TQDMProgressBar

from chggen.common.data_utils import get_scaler
from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_data.datamodule import CrystDataModule
from chggen.pl_modules.model_egnn import CHGGen

/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Dataset.
train_dataset = CHGNetDataset(
    path= '/home/xzdai/ceder_group/material_dircovery/chggen_old/data/perov_5/test_zpc.csv', # train.csv
    name = 'train_perov',
    prop_list = ['heat_all'],
)

val_dataset = CHGNetDataset(
    path= '/home/xzdai/ceder_group/material_dircovery/chggen_old/data/perov_5/test_zpc.csv', # val.csv
    name = 'val_perov',
    prop_list = ['heat_all'],
)

# Compute lattice scaler and save.
lattice_scaler = get_scaler(dataset = train_dataset)

100%|██████████| 50/50 [00:00<00:00, 200.85it/s]
/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:655: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  targets = torch.tensor([d[key] for d in data_list])
/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:619: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


In [7]:
datamodule = CrystDataModule(
    train_dataset = train_dataset,
    val_dataset = val_dataset,
    num_workers = 8,
    batch_size = 16,
)

model_hparams = {'latent_dim': 64, 'hidden_dim': 128, 
                'predict_property': True, 'property_dim': 1, # predict the multiple property 
                'load_pretrain': True, 'fc_num_layers': 1, 
                'sigma_F_begin': 10.0, 'sigma_F_end': 0.01, 
                'sigma_L_begin': 1.0, 'sigma_L_end': 0.01, 
                'type_sigma_begin': 5.0, 'type_sigma_end': 0.01,
                'max_atoms': 20, # should be larger than the training set.
                'num_noise_level': 5, 
                'lattice_scale_method': 'scale_length', 
                'cost_natom': 1.0, 'cost_latt': 10.0, 'cost_coord': 10.0, 'cost_type': 1.0, 'cost_lattice': 10.0, 'cost_composition': 1.0, 'cost_edge': 10.0, 'cost_property': 1.0,
                'beta': 0.01,
                'teacher_forcing_lattice': True,
                'teacher_forcing_max_epoch': 1000,
                'decoder': 'egnn'}

chggen = CHGGen(
    lattice_scaler = lattice_scaler, hparams_dict = model_hparams
)

device = torch.device('cpu')
checkpoint_path = "/home/xzdai/ceder_group/material_dircovery/chggen_old/test_models/perov/trainer_perov.ckpt"
chggen = chggen.load_from_checkpoint(checkpoint_path = checkpoint_path)
chggen.lattice_scaler = lattice_scaler
chggen.to(device = device)

CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


CHGGen(
  (mse_composition): MSELoss()
  (wnd): wND()
  (encoder3d): CHGNet_encoder(
    (composition_model): AtomRef(
      (fc): Linear(in_features=94, out_features=1, bias=False)
    )
    (graph_converter): CrystalGraphConverter(algorithm='legacy', atom_graph_cutoff=5, bond_graph_cutoff=3)
    (atom_embedding): AtomEmbedding(
      (embedding): Embedding(94, 64)
    )
    (bond_basis_expansion): BondEncoder(
      (rbf_expansion_ag): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
      (rbf_expansion_bg): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
    )
    (bond_embedding): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_ag): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_bg): Linear(in_features=9, out_features=64, bias=False)
    (angle_basis_expansion): AngleEncoder(
      (fourier_expansion): Fourier()
    )
    (angle_embedding): Linear(in_features=9, out_features=64, bias=False)
    (atom_con

In [8]:
data = next(iter(datamodule.train_dataloader()))
data

DataBatch(x=[80], edge_index=[2, 2892], crys_graph=[16], lattices=[16, 9], reduced_lattices=[16, 9], frac_coords=[80, 3], atom_types=[80], lengths=[16, 3], angles=[16, 3], num_atoms=[16], properties=[16, 1], batch=[80], ptr=[17])

In [ ]:
# Define the checkpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath= './test_models/perov/',
    filename='{epoch}',        # Save the checkpoint after every epoch
    save_top_k=-1,            # Set to -1 to save all checkpoints
    save_last=True,           # Save the last model too, useful for resuming
    every_n_train_steps=100,     # Save every epoch (assuming you're validating every epoch)
    verbose=True              # Print save messages for debugging
)

trainer = pl.Trainer(
    accelerator = "gpu", 
    devices = [0],
    max_epochs = 10,
    callbacks = [checkpoint_callback, TQDMProgressBar(refresh_rate = 1)],
    #  strategy = 'ddp_find_unused_parameters_true',  # multi-GPU training                    
)

trainer.fit(model = chggen, datamodule = datamodule)

ModelCheckpoint(save_last=True, save_top_k=-1, monitor=None) will duplicate the last checkpoint saved.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:67: UserWarning: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
  warning_cache.warn(
/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:617: UserWarning: Checkpoint directory /home/xzdai/ced

Sanity Checking: 0it [00:00, ?it/s]

/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/torch_geometric/deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00,  1.58it/s]

/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:630: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
/home/xzdai/ceder_group/material_dircovery/chggen_old/chggen/common/data_utils.py:626: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/pytorch_lightning/utilities/data.py:76: UserWarning: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 80. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
  warning_cache.warn(


...

Epoch 0: 100%|██████████| 4/4 [00:01<00:00,  3.36it/s, v_num=81, train_loss_step=2.87e+4, train_natom_loss_step=2.770, train_lattice_loss_step=0.100, train_latt_loss_step=1.75e+3, train_coord_loss_step=1.12e+3, train_kld_loss_step=3.040, train_composition_loss_step=4.540]

/home/xzdai/anaconda3/envs/chggen/lib/python3.8/site-packages/pytorch_lightning/utilities/data.py:76: UserWarning: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 10. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
  warning_cache.warn(


Epoch 0: 100%|██████████| 4/4 [00:01<00:00,  2.11it/s, v_num=81, train_loss_step=2.87e+4, train_natom_loss_step=2.770, train_lattice_loss_step=0.100, train_latt_loss_step=1.75e+3, train_coord_loss_step=1.12e+3, train_kld_loss_step=3.040, train_composition_loss_step=4.540, val_loss=4.75e+3, val_natom_loss=2.910, val_lattice_loss=0.495, val_latt_loss=353.0, val_coord_loss=122.0, val_kld_loss=3.560, val_composition_loss=4.540, val_property_loss=2.320, val_natom_accuracy=0.160, val_lengths_mard=0.062, val_angles_mae=2.03e-7, val_volumes_mard=0.186, train_loss_epoch=1.67e+4, train_natom_loss_epoch=2.960, train_lattice_loss_epoch=0.543, train_latt_loss_epoch=1.35e+3, train_coord_loss_epoch=312.0, train_kld_loss_epoch=2.740, train_composition_loss_epoch=4.540]****************************************************************************************************
Epoch 0 - loss: 28660.8027
Epoch 0 - num_atom_loss: 2.7692
Epoch 0 - lattice_loss: 0.1002
Epoch 0 - latt_loss: 1747.3640
Epoch 0 - coord

`Trainer.fit` stopped: `max_epochs=10` reached.


****************************************************************************************************
Epoch 9 - loss: 2699.0872
Epoch 9 - num_atom_loss: 0.0066
Epoch 9 - lattice_loss: 0.0751
Epoch 9 - latt_loss: 102.7110
Epoch 9 - coord_loss: 166.6352
Epoch 9 - type_loss: 0.0000
Epoch 9 - kld_loss: 41.8338
Epoch 9 - composition_loss: 4.3639
Epoch 9: 100%|██████████| 4/4 [00:02<00:00,  1.81it/s, v_num=81, train_loss_step=2.7e+3, train_natom_loss_step=0.00655, train_lattice_loss_step=0.0751, train_latt_loss_step=103.0, train_coord_loss_step=167.0, train_kld_loss_step=41.80, train_composition_loss_step=4.360, val_loss=466.0, val_natom_loss=0.0128, val_lattice_loss=0.104, val_latt_loss=23.20, val_coord_loss=23.40, val_kld_loss=40.60, val_composition_loss=4.290, val_property_loss=0.280, val_natom_accuracy=1.000, val_lengths_mard=0.0276, val_angles_mae=1.02e-7, val_volumes_mard=0.0784, train_loss_epoch=713.0, train_natom_loss_epoch=0.0358, train_lattice_loss_epoch=0.106, train_latt_loss_epoch